# Data Quality Validation

**Purpose**: Run all 5 baseline validation checks and write results to `dq_results` and `dq_exception_rows`

**Replicates Excel Workbook's "Check" Sheet**:
- DATE_001 (warning): Report date within 5 working days of month end
- MV_001 (exception): Holdings × Price × FX ≈ Bid Value (tolerance ±£1 or ±0.1%)
- VAL_001 (exception): Policy must have cash and/or stock value
- MAP_001 (exception): Securities must have identifier (cash exempt)
- POP_001 (exception): Policy must map to IH policy ID

**Input**: `individual_dfm_consolidated` (Stage 2 output)

**Output**: `dq_results`, `dq_exception_rows`

In [ ]:
# ========== Establish Workspace Context ==========
# CRITICAL: Required when notebook is in a subfolder
# This ensures Fabric's lakehouse context is properly initialized
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")
    print("[ACTION] Ensure lakehouse is attached in Fabric notebook UI")

# ========== Parameters ==========
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by orchestrator

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    pass

print(f"[VALIDATION] Starting data quality checks")
print(f"  Period: {period}")
print(f"  Run ID: {run_id}")

In [ ]:
# ========== Imports ==========
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from datetime import datetime, timedelta
import json

print("[VALIDATION] Imports complete")

## Step 1: Load Configuration and Data

Load rules configuration and Stage 2 data.

In [ ]:
# ========== Load Configuration ==========
print("\n[STEP 1] Loading configuration and data...")
print("=" * 70)

# Load rules configuration
try:
    with open("/lakehouse/default/Files/config/rules_config.json", "r") as f:
        rules_config = json.load(f)
    print(f"  Rules config loaded: {len(rules_config.get('validation_rules', []))} rules")
except Exception as e:
    print(f"  [WARNING] Could not load rules_config.json: {e}")
    rules_config = {"validation_rules": []}

# Extract rule configurations
rules = {rule['rule_id']: rule for rule in rules_config.get('validation_rules', [])}
date_rule = rules.get('DATE_001', {'enabled': True})
mv_rule = rules.get('MV_001', {'enabled': True, 'tolerance_pct': 0.1, 'tolerance_abs': 1.0})
val_rule = rules.get('VAL_001', {'enabled': True})
map_rule = rules.get('MAP_001', {'enabled': True})
pop_rule = rules.get('POP_001', {'enabled': False})  # Disabled by default

print(f"  DATE_001: {'enabled' if date_rule.get('enabled') else 'disabled'}")
print(f"  MV_001: {'enabled' if mv_rule.get('enabled') else 'disabled'}")
print(f"  VAL_001: {'enabled' if val_rule.get('enabled') else 'disabled'}")
print(f"  MAP_001: {'enabled' if map_rule.get('enabled') else 'disabled'}")
print(f"  POP_001: {'enabled' if pop_rule.get('enabled') else 'disabled'}")

# Load Stage 2 data
try:
    stage2_data = (
        spark.table("individual_dfm_consolidated")
        .filter((F.col("period") == period) & (F.col("run_id") == run_id))
    )
    stage2_count = stage2_data.count()
    print(f"\n  Stage 2 data: {stage2_count} rows")
except Exception as e:
    print(f"  [ERROR] Could not read individual_dfm_consolidated: {e}")
    mssparkutils.notebook.exit("FAILED")

if stage2_count == 0:
    print("  [ERROR] No Stage 2 data found for this period/run_id.")
    mssparkutils.notebook.exit("NO_DATA")

print("\n[STEP 1] Configuration and data loaded successfully")

## Step 2: DATE_001 - Report Date Timeliness

**Rule**: Report date should be within 5 working days of month end

**Severity**: Warning

In [ ]:
# ========== DATE_001: Report Date Timeliness ==========
print("\n[CHECK] DATE_001: Report date timeliness...")
print("=" * 70)

date_001_results = []
date_001_exceptions = []

if date_rule.get('enabled'):
    # Calculate month end from period (YYYY-MM)
    from dateutil.relativedelta import relativedelta
    year, month = map(int, period.split('-'))
    month_end = datetime(year, month, 1) + relativedelta(months=1) - timedelta(days=1)
    
    # 5 working days = approximately 7 calendar days (rough estimate)
    timeliness_threshold = month_end + timedelta(days=7)
    
    print(f"  Month end: {month_end.date()}")
    print(f"  Timeliness threshold (approx): {timeliness_threshold.date()}")
    
    # Check if any report dates exceed threshold
    late_reports = stage2_data.filter(
        F.col("report_date").isNotNull() &
        (F.col("report_date") > F.lit(timeliness_threshold.date()))
    )
    
    late_count = late_reports.count()
    print(f"  Late report dates: {late_count} rows")
    
    # Aggregate by DFM
    late_by_dfm = late_reports.groupBy("dfm_id").count()
    
    for row in late_by_dfm.collect():
        date_001_results.append({
            "check_id": "DATE_001",
            "severity": "warning",
            "status": "fail" if row['count'] > 0 else "pass",
            "metric_value": float(row['count']),
            "threshold": 0.0,
            "details_json": json.dumps({"dfm_id": row['dfm_id'], "late_count": row['count']})
        })
    
    # Capture exception rows
    if late_count > 0:
        for row in late_reports.select("dfm_id", "source_file", "source_row_id", "report_date").limit(100).collect():
            date_001_exceptions.append({
                "check_id": "DATE_001",
                "dfm_id": row['dfm_id'],
                "source_file": row['source_file'],
                "source_row_id": row['source_row_id'],
                "failure_reason": f"Report date {row['report_date']} exceeds timeliness threshold {timeliness_threshold.date()}"
            })
    
    print(f"  DATE_001: {'PASS' if late_count == 0 else 'FAIL (WARNING)'}")
else:
    print("  DATE_001: SKIPPED (disabled)")

print("\n[CHECK] DATE_001 complete")

## Step 3: MV_001 - Market Value Reconciliation

**Rule**: Holdings × Price × FX ≈ Bid Value (tolerance ±£1 or ±0.1%)

**Severity**: Exception

In [ ]:
# ========== MV_001: Market Value Reconciliation ==========
print("\n[CHECK] MV_001: Market value reconciliation...")
print("=" * 70)

mv_001_results = []
mv_001_exceptions = []

if mv_rule.get('enabled'):
    tolerance_pct = mv_rule.get('tolerance_pct', 0.1) / 100.0  # Convert to decimal
    tolerance_abs = mv_rule.get('tolerance_abs', 1.0)
    
    print(f"  Tolerance: ±{tolerance_pct * 100}% or ±£{tolerance_abs}")
    
    # Calculate expected value: holding × price × fx_rate
    evaluable = stage2_data.filter(
        F.col("holding").isNotNull() &
        F.col("local_bid_price").isNotNull() &
        F.col("fx_rate").isNotNull() &
        F.col("bid_value_gbp").isNotNull() &
        (F.col("bid_value_gbp") != 0.0)
    )
    
    evaluable = evaluable.withColumn(
        "_expected_value",
        F.col("holding") * F.col("local_bid_price") * F.col("fx_rate")
    )
    
    evaluable = evaluable.withColumn(
        "_abs_diff",
        F.abs(F.col("_expected_value") - F.col("bid_value_gbp"))
    )
    
    evaluable = evaluable.withColumn(
        "_pct_diff",
        F.when(
            F.col("bid_value_gbp") != 0.0,
            F.col("_abs_diff") / F.abs(F.col("bid_value_gbp"))
        ).otherwise(F.lit(None))
    )
    
    # Identify failures: exceed both absolute AND percentage tolerance
    mv_failures = evaluable.filter(
        (F.col("_abs_diff") > tolerance_abs) &
        (F.col("_pct_diff") > tolerance_pct)
    )
    
    evaluable_count = evaluable.count()
    failure_count = mv_failures.count()
    
    print(f"  Evaluable rows: {evaluable_count}")
    print(f"  MV reconciliation failures: {failure_count}")
    print(f"  Failure rate: {failure_count / evaluable_count * 100:.1f}%" if evaluable_count > 0 else "  Failure rate: N/A")
    
    # Aggregate results by DFM
    failure_by_dfm = mv_failures.groupBy("dfm_id").count()
    
    for row in failure_by_dfm.collect():
        mv_001_results.append({
            "check_id": "MV_001",
            "severity": "exception",
            "status": "fail",
            "metric_value": float(row['count']),
            "threshold": tolerance_pct * 100.0,
            "details_json": json.dumps({"dfm_id": row['dfm_id'], "failure_count": row['count'], "evaluable": evaluable_count})
        })
    
    # If no failures, still record pass
    if failure_count == 0:
        mv_001_results.append({
            "check_id": "MV_001",
            "severity": "exception",
            "status": "pass",
            "metric_value": 0.0,
            "threshold": tolerance_pct * 100.0,
            "details_json": json.dumps({"evaluable": evaluable_count})
        })
    
    # Capture exception rows
    if failure_count > 0:
        for row in mv_failures.select("dfm_id", "source_file", "source_row_id", "bid_value_gbp", "_expected_value", "_abs_diff", "_pct_diff").limit(100).collect():
            mv_001_exceptions.append({
                "check_id": "MV_001",
                "dfm_id": row['dfm_id'],
                "source_file": row['source_file'],
                "source_row_id": row['source_row_id'],
                "failure_reason": f"MV mismatch: actual £{row['bid_value_gbp']:.2f}, expected £{row['_expected_value']:.2f}, diff £{row['_abs_diff']:.2f} ({row['_pct_diff']*100:.2f}%)"
            })
    
    print(f"  MV_001: {'PASS' if failure_count == 0 else 'FAIL (EXCEPTION)'}")
else:
    print("  MV_001: SKIPPED (disabled)")

print("\n[CHECK] MV_001 complete")

## Step 4: VAL_001 - Policy Value Check

**Rule**: Policy must have cash and/or stock value (not all zeros)

**Severity**: Exception

In [ ]:
# ========== VAL_001: Policy Value Check ==========
print("\n[CHECK] VAL_001: Policy value check...")
print("=" * 70)

val_001_results = []
val_001_exceptions = []

if val_rule.get('enabled'):
    # Aggregate by policy
    policy_totals = stage2_data.groupBy("dfm_id", "policyholder_number").agg(
        F.sum(F.coalesce(F.col("bid_value_gbp"), F.lit(0.0))).alias("total_bid_value"),
        F.sum(F.coalesce(F.col("cash_value_gbp"), F.lit(0.0))).alias("total_cash_value")
    )
    
    # Identify policies with zero value
    zero_value_policies = policy_totals.filter(
        (F.col("total_bid_value") == 0.0) &
        (F.col("total_cash_value") == 0.0)
    )
    
    total_policies = policy_totals.count()
    zero_count = zero_value_policies.count()
    
    print(f"  Total policies: {total_policies}")
    print(f"  Zero-value policies: {zero_count}")
    print(f"  Failure rate: {zero_count / total_policies * 100:.1f}%" if total_policies > 0 else "  Failure rate: N/A")
    
    # Aggregate by DFM
    zero_by_dfm = zero_value_policies.groupBy("dfm_id").count()
    
    for row in zero_by_dfm.collect():
        val_001_results.append({
            "check_id": "VAL_001",
            "severity": "exception",
            "status": "fail",
            "metric_value": float(row['count']),
            "threshold": 0.0,
            "details_json": json.dumps({"dfm_id": row['dfm_id'], "zero_value_policies": row['count']})
        })
    
    # If no failures, record pass
    if zero_count == 0:
        val_001_results.append({
            "check_id": "VAL_001",
            "severity": "exception",
            "status": "pass",
            "metric_value": 0.0,
            "threshold": 0.0,
            "details_json": json.dumps({"total_policies": total_policies})
        })
    
    # Capture exception rows (join back to get source_row_id)
    if zero_count > 0:
        exception_policies = zero_value_policies.join(
            stage2_data,
            ["dfm_id", "policyholder_number"],
            "inner"
        ).select("dfm_id", "source_file", "source_row_id", "policyholder_number")
        
        for row in exception_policies.limit(100).collect():
            val_001_exceptions.append({
                "check_id": "VAL_001",
                "dfm_id": row['dfm_id'],
                "source_file": row['source_file'],
                "source_row_id": row['source_row_id'],
                "failure_reason": f"Policy {row['policyholder_number']} has zero cash and stock value"
            })
    
    print(f"  VAL_001: {'PASS' if zero_count == 0 else 'FAIL (EXCEPTION)'}")
else:
    print("  VAL_001: SKIPPED (disabled)")

print("\n[CHECK] VAL_001 complete")

## Step 5: MAP_001 - Identifier Completeness

**Rule**: Non-cash securities must have identifier (ISIN or SEDOL)

**Severity**: Exception

**Note**: Cash positions (id_type = "Undertaking - Specific") are exempt

In [ ]:
# ========== MAP_001: Identifier Completeness ==========
print("\n[CHECK] MAP_001: Identifier completeness...")
print("=" * 70)

map_001_results = []
map_001_exceptions = []

if map_rule.get('enabled'):
    # Identify rows missing identifiers (excluding cash)
    missing_identifiers = stage2_data.filter(
        (F.col("security_code").isNull() | (F.trim(F.col("security_code")) == "")) &
        (F.col("isin").isNull() | (F.trim(F.col("isin")) == "")) &
        (F.col("sedol").isNull() | (F.trim(F.col("sedol")) == "")) &
        ~F.lower(F.col("id_type")).contains("undertaking")
    )
    
    missing_count = missing_identifiers.count()
    total_non_cash = stage2_data.filter(~F.lower(F.col("id_type")).contains("undertaking")).count()
    
    print(f"  Non-cash securities: {total_non_cash}")
    print(f"  Missing identifiers: {missing_count}")
    print(f"  Failure rate: {missing_count / total_non_cash * 100:.1f}%" if total_non_cash > 0 else "  Failure rate: N/A")
    
    # Aggregate by DFM
    missing_by_dfm = missing_identifiers.groupBy("dfm_id").count()
    
    for row in missing_by_dfm.collect():
        map_001_results.append({
            "check_id": "MAP_001",
            "severity": "exception",
            "status": "fail",
            "metric_value": float(row['count']),
            "threshold": 0.0,
            "details_json": json.dumps({"dfm_id": row['dfm_id'], "missing_identifiers": row['count']})
        })
    
    # If no failures, record pass
    if missing_count == 0:
        map_001_results.append({
            "check_id": "MAP_001",
            "severity": "exception",
            "status": "pass",
            "metric_value": 0.0,
            "threshold": 0.0,
            "details_json": json.dumps({"non_cash_securities": total_non_cash})
        })
    
    # Capture exception rows
    if missing_count > 0:
        for row in missing_identifiers.select("dfm_id", "source_file", "source_row_id", "asset_name", "id_type").limit(100).collect():
            map_001_exceptions.append({
                "check_id": "MAP_001",
                "dfm_id": row['dfm_id'],
                "source_file": row['source_file'],
                "source_row_id": row['source_row_id'],
                "failure_reason": f"Missing identifier for {row['asset_name']} (id_type: {row['id_type']})"
            })
    
    print(f"  MAP_001: {'PASS' if missing_count == 0 else 'FAIL (EXCEPTION)'}")
else:
    print("  MAP_001: SKIPPED (disabled)")

print("\n[CHECK] MAP_001 complete")

## Step 6: POP_001 - Policy Mapping Check

**Rule**: Policy must map to IH policy ID

**Severity**: Exception

**Note**: This check is typically disabled in PoC (relies on complete policy mapping)

In [ ]:
# ========== POP_001: Policy Mapping Check ==========
print("\n[CHECK] POP_001: Policy mapping check...")
print("=" * 70)

pop_001_results = []
pop_001_exceptions = []

if pop_rule.get('enabled'):
    # Identify rows with missing policyholder_number
    unmapped_policies = stage2_data.filter(
        F.col("policyholder_number").isNull() |
        (F.trim(F.col("policyholder_number")) == "")
    )
    
    unmapped_count = unmapped_policies.count()
    
    print(f"  Total rows: {stage2_count}")
    print(f"  Unmapped policies: {unmapped_count}")
    print(f"  Failure rate: {unmapped_count / stage2_count * 100:.1f}%" if stage2_count > 0 else "  Failure rate: N/A")
    
    # Aggregate by DFM
    unmapped_by_dfm = unmapped_policies.groupBy("dfm_id").count()
    
    for row in unmapped_by_dfm.collect():
        pop_001_results.append({
            "check_id": "POP_001",
            "severity": "exception",
            "status": "fail",
            "metric_value": float(row['count']),
            "threshold": 0.0,
            "details_json": json.dumps({"dfm_id": row['dfm_id'], "unmapped_count": row['count']})
        })
    
    # If no failures, record pass
    if unmapped_count == 0:
        pop_001_results.append({
            "check_id": "POP_001",
            "severity": "exception",
            "status": "pass",
            "metric_value": 0.0,
            "threshold": 0.0,
            "details_json": json.dumps({"total_rows": stage2_count})
        })
    
    # Capture exception rows
    if unmapped_count > 0:
        for row in unmapped_policies.select("dfm_id", "source_file", "source_row_id").limit(100).collect():
            pop_001_exceptions.append({
                "check_id": "POP_001",
                "dfm_id": row['dfm_id'],
                "source_file": row['source_file'],
                "source_row_id": row['source_row_id'],
                "failure_reason": "Policy not mapped to IH policy ID"
            })
    
    print(f"  POP_001: {'PASS' if unmapped_count == 0 else 'FAIL (EXCEPTION)'}")
else:
    print("  POP_001: SKIPPED (disabled)")

print("\n[CHECK] POP_001 complete")

## Step 7: Write Results to dq_results Table

Persist check results.

In [ ]:
# ========== Write DQ Results ==========
print("\n[STEP 7] Writing to dq_results table...")
print("=" * 70)

all_results = date_001_results + mv_001_results + val_001_results + map_001_results + pop_001_results

if len(all_results) > 0:
    results_df = spark.createDataFrame(all_results)
    results_df = results_df \
        .withColumn("period", F.lit(period)) \
        .withColumn("run_id", F.lit(run_id)) \
        .withColumn("evaluated_at", F.current_timestamp())
    
    try:
        results_df.write.mode("append").saveAsTable("dq_results")
        print(f"  ✓ Wrote {len(all_results)} check results to dq_results")
    except Exception as e:
        print(f"  [ERROR] Failed to write to dq_results: {e}")
else:
    print("  No results to write (all checks disabled or skipped)")

print("\n[STEP 7] DQ results written")

## Step 8: Write Exception Rows to dq_exception_rows Table

Persist failing rows for investigation.

In [ ]:
# ========== Write Exception Rows ==========
print("\n[STEP 8] Writing to dq_exception_rows table...")
print("=" * 70)

all_exceptions = date_001_exceptions + mv_001_exceptions + val_001_exceptions + map_001_exceptions + pop_001_exceptions

if len(all_exceptions) > 0:
    exceptions_df = spark.createDataFrame(all_exceptions)
    exceptions_df = exceptions_df \
        .withColumn("period", F.lit(period)) \
        .withColumn("run_id", F.lit(run_id)) \
        .withColumn("created_at", F.current_timestamp())
    
    try:
        exceptions_df.write.mode("append").saveAsTable("dq_exception_rows")
        print(f"  ✓ Wrote {len(all_exceptions)} exception rows to dq_exception_rows")
    except Exception as e:
        print(f"  [ERROR] Failed to write to dq_exception_rows: {e}")
else:
    print("  No exception rows to write (all checks passed)")

print("\n[STEP 8] Exception rows written")

## Summary

Final validation summary.

In [ ]:
# ========== Summary ==========
print("\n" + "=" * 70)
print("DATA QUALITY VALIDATION COMPLETE")
print("=" * 70)

print(f"\nPeriod: {period}")
print(f"Run ID: {run_id}")

print(f"\nChecks Run:")
for check_id in ["DATE_001", "MV_001", "VAL_001", "MAP_001", "POP_001"]:
    check_results = [r for r in all_results if r['check_id'] == check_id]
    if check_results:
        status = check_results[0]['status']
        severity = check_results[0]['severity']
        print(f"  {check_id}: {status.upper()} ({severity})")
    else:
        print(f"  {check_id}: SKIPPED")

fail_count = len([r for r in all_results if r['status'] == 'fail' and r['severity'] == 'exception'])
warn_count = len([r for r in all_results if r['status'] == 'fail' and r['severity'] == 'warning'])

print(f"\nOverall Status:")
print(f"  Exceptions: {fail_count}")
print(f"  Warnings: {warn_count}")

if fail_count > 0:
    print(f"\n  ⚠ VALIDATION FAILED: {fail_count} exception(s) found")
    print(f"  Review dq_exception_rows table for details")
elif warn_count > 0:
    print(f"\n  ⚠ VALIDATION PASSED WITH WARNINGS: {warn_count} warning(s)")
else:
    print(f"\n  ✓ VALIDATION PASSED: All checks passed")

print("\n" + "=" * 70)

# Exit with status
if fail_count > 0:
    mssparkutils.notebook.exit("FAIL")
else:
    mssparkutils.notebook.exit("OK")